In [0]:
import json
import random
from datetime import datetime, timedelta

dbutils.widgets.text("catalog_name", "dbr_dev")
dbutils.widgets.text("storage_account", "dlspl21databricks")
dbutils.widgets.text("container", "janvander0912")
dbutils.widgets.text("volume", "raw_data")

catalog_name = dbutils.widgets.get("catalog_name")
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")
volume_name = dbutils.widgets.get("volume")

target_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{volume_name}/truck_stream/"

In [0]:
# cleaning folder before test
dbutils.fs.rm(target_path, True)
dbutils.fs.mkdirs(target_path)

In [0]:
cargos = {
    "VACCINES":    {"min": -20.0, "max": -15.0, "current": -17.5}, 
    "ICE_CREAM":   {"min": -25.0, "max": -18.0, "current": -21.0}, 
    "FRESH_MEAT":  {"min":   0.0, "max":   4.0, "current":   2.0}, 
    "CHOCOLATE":   {"min":  15.0, "max":  18.0, "current":  16.5}  
}

cargo_names = list(cargos.keys())
start_time = datetime.now() - timedelta(hours=10)


print("Generating 1000 files")
for i in range(1, 1001):
    cargo_type = random.choice(cargo_names)
    bounds = cargos[cargo_type]

    temp_change = random.uniform(-0.3, 0.3)
    bounds["current"] = round(bounds["current"] + temp_change, 2)
    current_temp = bounds["current"]

    if current_temp < bounds["min"] or current_temp > bounds["max"]:
        status = "WARNING"
    else:
        status = "OK"

    timestamp_str = (start_time + timedelta(seconds=i*30)).isoformat()

    data = {
        "measurement_id": i,
        "cargo_type": cargo_type,
        "current_temp": current_temp,
        "min_temp": bounds["min"],
        "max_temp": bounds["max"],
        "status": status,
        "timestamp": timestamp_str  
    }

    if 800 < i <= 900:
        data["door_open"] = random.choice(([True, False, False, False]))

    if i > 900:
        data["current_temp"] = f"SENSOR_FAIL_ERR_{random.randint(100, 500)}"
        data["status"] = "ERROR"
        data["timestamp"] = False

    file_name = f"measurement_{i:04d}.json"
    dbutils.fs.put(f"{target_path}{file_name}", json.dumps(data), True)

    if i % 200 == 0:
        print(f"Generated and saved: {i}/1000 files")


print("Success! Generated 1000 files ready for streaming")
